# Project 3: Are late-night car crashes more dangerous?

Driving through New York City at 3 AM feels completely different from driving at 3 PM. The streets are empty, so you'd think it would be safer. But there are also fewer people paying attention, more people who've been drinking, and more room to speed.

In this project, I'll use NYC's record of every police-reported car crash to test whether crashes that happen in the middle of the night are more dangerous than crashes during the day.

## Prompt

- **Dataset(s) to be used:** [Motor Vehicle Collisions - Crashes](https://data.cityofnewyork.us/Public-Safety/Motor-Vehicle-Collisions-Crashes/h9gi-nx95) (NYC Open Data, from the NYPD), filtered to crashes in 2024
- **Analysis question:** Are crashes between midnight and 6 AM more likely to injure or kill someone than crashes during the rest of the day? If so, what might explain the difference?
- **Columns that will (likely) be used:**
  - `crash_time`
  - `number_of_persons_injured`
  - `number_of_persons_killed`
  - `number_of_pedestrians_injured`, `number_of_cyclist_injured`, `number_of_motorist_injured`
  - `contributing_factor_vehicle_1`
- **Hypothesis**: Late-night crashes are more dangerous than daytime crashes: a larger share of them injure someone, and a larger share of them are fatal.

## The data

The full crash dataset has over two million rows going back to 2012, with lots of columns I don't need (like exact locations and street names). To keep things manageable, I downloaded only 2024 and only the columns I'm using, with this link:

`https://data.cityofnewyork.us/resource/h9gi-nx95.csv?$select=collision_id,crash_date,crash_time,borough,number_of_persons_injured,number_of_persons_killed,number_of_pedestrians_injured,number_of_pedestrians_killed,number_of_cyclist_injured,number_of_cyclist_killed,number_of_motorist_injured,number_of_motorist_killed,contributing_factor_vehicle_1&$where=crash_date between '2024-01-01T00:00:00' and '2024-12-31T23:59:59'&$limit=200000`

and saved it as `crashes.csv`.

## Setup

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

# ensure the visualizations render properly across VSCode, Jupyter Book, etc.
# https://plotly.com/python/renderers/
pio.renderers.default = "notebook_connected+plotly_mimetype"

In [ ]:
crashes = pd.read_csv("crashes.csv")
crashes.head()

In [ ]:
crashes.info()

In [ ]:
print(f"Number of crashes: {len(crashes):,}")

Each row is one crash. The injury and death counts tell us how many people were hurt or killed, and who they were (pedestrians, cyclists, or people in vehicles).

## Cleaning the data

### What time did each crash happen?

`crash_time` is stored as text, like `"14:35"`. Let's look at which times show up most often.

In [ ]:
crashes["crash_time"].value_counts().head(10)

Two things jump out:

1. **Times on the hour (like 15:00) are very common.** People probably round when reporting a crash, so I'll only rely on the _hour_, not the exact minute.
2. **Midnight (00:00) is by far the most common time**, way more than any other minute. It's very unlikely that so many crashes happen at exactly midnight. More likely, 00:00 is filled in when the real time is unknown.

Since I'm specifically studying late-night crashes, those fake midnight times would mess up my results, so I'll set them aside.

In [ ]:
unknown_time = crashes["crash_time"] == "00:00"
print(
    f"Crashes recorded at exactly midnight: {unknown_time.sum():,} ({unknown_time.mean():.1%})"
)

crashes = crashes[~unknown_time].copy()
crashes["hour"] = crashes["crash_time"].str.split(":").str[0].astype(int)

### Was anyone hurt?

I'll add two yes/no columns:

- `anyone_hurt`: at least one person was injured or killed
- `fatal`: at least one person was killed

In [ ]:
crashes["anyone_hurt"] = (crashes["number_of_persons_injured"] > 0) | (
    crashes["number_of_persons_killed"] > 0
)
crashes["fatal"] = crashes["number_of_persons_killed"] > 0

crashes[["anyone_hurt", "fatal"]].mean()

### Defining "late night"

I'll call any crash between midnight and 5:59 AM a **late-night** crash, and everything else **daytime** (which includes the evening).

In [ ]:
NIGHT_START = 0
NIGHT_END = 6

is_night = (crashes["hour"] >= NIGHT_START) & (crashes["hour"] < NIGHT_END)
crashes["period"] = is_night.map(
    {True: "Late night (12-6 AM)", False: "Daytime (6 AM-12 AM)"}
)

crashes["period"].value_counts()

## Part 1: How dangerous are crashes at each hour?

For each hour of the day, I'll calculate:

- the share of crashes that hurt someone
- the number of fatal crashes per 1,000 crashes (fatal crashes are rare, so a percentage would be a tiny number)

In [ ]:
by_hour = crashes.groupby("hour").agg(
    crashes=("collision_id", "count"),
    hurt_rate=("anyone_hurt", "mean"),
    fatal_rate=("fatal", "mean"),
)
by_hour["hurt_pct"] = by_hour["hurt_rate"] * 100
by_hour["fatal_per_1k"] = by_hour["fatal_rate"] * 1_000
by_hour = by_hour.reset_index()
by_hour

First, let's see how many crashes happen at each hour, so we know how much data each hour is based on.

In [ ]:
fig = px.bar(
    by_hour,
    x="hour",
    y="crashes",
    title="Number of crashes by hour of day (2024)",
    labels={"hour": "Hour of day (0 = midnight)", "crashes": "Number of crashes"},
)
fig.show()

As you'd expect, far fewer crashes happen in the middle of the night, simply because there are fewer cars on the road. That's why I'm comparing _rates_ (what share of crashes are serious) rather than raw counts.

### Are late-night crashes more likely to hurt someone?

In [ ]:
fig = px.line(
    by_hour,
    x="hour",
    y="hurt_pct",
    markers=True,
    title="Share of crashes that injured or killed someone, by hour (2024)",
    labels={
        "hour": "Hour of day (0 = midnight)",
        "hurt_pct": "Crashes with an injury or death (%)",
    },
)
fig.add_vrect(
    x0=NIGHT_START - 0.5,
    x1=NIGHT_END - 0.5,
    fillcolor="gray",
    opacity=0.15,
    line_width=0,
)
fig.show()

This is the **opposite** of what I expected. The shaded late-night hours actually have the _lowest_ share of crashes where someone gets hurt. The riskiest time for injuries is the early evening.

### Are late-night crashes more likely to kill someone?

In [ ]:
fig = px.line(
    by_hour,
    x="hour",
    y="fatal_per_1k",
    markers=True,
    title="Fatal crashes per 1,000 crashes, by hour (2024)",
    labels={
        "hour": "Hour of day (0 = midnight)",
        "fatal_per_1k": "Fatal crashes per 1,000 crashes",
    },
)
fig.add_vrect(
    x0=NIGHT_START - 0.5,
    x1=NIGHT_END - 0.5,
    fillcolor="gray",
    opacity=0.15,
    line_width=0,
)
fig.show()

Now the late-night hours stand out in the other direction: crashes in the middle of the night are much more likely to be deadly. The line is bumpy because fatal crashes are rare, so each hour only has a handful. Grouping the hours into two periods gives a more reliable comparison:

In [ ]:
by_period = crashes.groupby("period").agg(
    crashes=("collision_id", "count"),
    hurt_rate=("anyone_hurt", "mean"),
    fatal_crashes=("fatal", "sum"),
    fatal_rate=("fatal", "mean"),
)
by_period["hurt_pct"] = by_period["hurt_rate"] * 100
by_period["fatal_per_1k"] = by_period["fatal_rate"] * 1_000
by_period = by_period[["crashes", "hurt_pct", "fatal_crashes", "fatal_per_1k"]]
by_period

In [ ]:
night = by_period.loc["Late night (12-6 AM)"]
day = by_period.loc["Daytime (6 AM-12 AM)"]

print(
    f"Share of crashes that hurt someone: {night['hurt_pct']:.1f}% at night vs. {day['hurt_pct']:.1f}% during the day"
)
print(
    f"Late-night crashes are {night['fatal_per_1k'] / day['fatal_per_1k']:.1f} times as likely to be fatal"
)

So my hypothesis is **half right**:

- ❌ Late-night crashes are _not_ more likely to injure someone.
- ✅ Late-night crashes _are_ much more likely to kill someone.

Why would that be? Two follow-up questions:

1. Who is getting hurt at different times of day?
2. What's causing the late-night crashes?

## Part 2: Who gets hurt?

The dataset splits injuries into pedestrians, cyclists, and motorists (people inside vehicles). Right now each of those is a separate column. To chart them together, I'll [reshape](https://pandas.pydata.org/docs/user_guide/reshaping.html) the data so that there's one row per hour per type of road user.

In [ ]:
injury_columns = {
    "number_of_pedestrians_injured": "Pedestrians",
    "number_of_cyclist_injured": "Cyclists",
    "number_of_motorist_injured": "Motorists",
}

injuries_by_hour = crashes.groupby("hour")[list(injury_columns)].sum()
injuries_by_hour = injuries_by_hour.rename(columns=injury_columns)

# turn the counts into injuries per 100 crashes
crashes_per_hour = crashes.groupby("hour").size()
injuries_per_100 = injuries_by_hour.div(crashes_per_hour, axis="index") * 100

# reshape from wide (one column per road user) to long (one row per hour per road user)
injuries_long = injuries_per_100.reset_index().melt(
    id_vars="hour", var_name="road_user", value_name="injured_per_100"
)
injuries_long.head()

In [ ]:
fig = px.line(
    injuries_long,
    x="hour",
    y="injured_per_100",
    color="road_user",
    markers=True,
    title="People injured per 100 crashes, by hour and type of road user (2024)",
    labels={
        "hour": "Hour of day (0 = midnight)",
        "injured_per_100": "People injured per 100 crashes",
        "road_user": "Road user",
    },
)
fig.add_vrect(
    x0=NIGHT_START - 0.5,
    x1=NIGHT_END - 0.5,
    fillcolor="gray",
    opacity=0.15,
    line_width=0,
)
fig.show()

This helps explain the first surprise. During the day, a big share of injuries are **pedestrians and cyclists**, who are easily hurt by even a slow-moving car. In the middle of the night there are far fewer people walking or biking, so pedestrian and cyclist injuries drop. Motorist injuries, on the other hand, stay high at night.

## Part 3: What's behind late-night crashes?

For each crash, the police officer records the main contributing factor. I'll compare how often each factor shows up at night versus during the day.

In [ ]:
def factor_shares(df, label):
    """Return the share of crashes for each contributing factor, as a percentage."""
    shares = df["contributing_factor_vehicle_1"].value_counts(normalize=True) * 100
    return shares.rename(label).reset_index()


night_factors = factor_shares(crashes[is_night], "night_pct")
day_factors = factor_shares(crashes[~is_night], "day_pct")

factors = pd.merge(night_factors, day_factors, on="contributing_factor_vehicle_1")
factors.head(10)

`Unspecified` means the officer didn't record a cause, so it doesn't tell us anything. I'll drop it, along with very rare factors whose percentages would be unreliable. Then I'll calculate how many times more common each factor is at night.

In [ ]:
MIN_NIGHT_CRASHES = 100

night_counts = crashes[is_night]["contributing_factor_vehicle_1"].value_counts()
common_factors = night_counts[night_counts >= MIN_NIGHT_CRASHES].index

factors = factors[
    factors["contributing_factor_vehicle_1"].isin(common_factors)
    & (factors["contributing_factor_vehicle_1"] != "Unspecified")
].copy()
factors["night_vs_day"] = factors["night_pct"] / factors["day_pct"]
factors = factors.sort_values("night_vs_day", ascending=False)
factors

In [ ]:
fig = px.bar(
    factors,
    x="night_vs_day",
    y="contributing_factor_vehicle_1",
    orientation="h",
    title="How much more common is each crash cause at night than during the day? (2024)",
    labels={
        "night_vs_day": "Share of late-night crashes ÷ share of daytime crashes",
        "contributing_factor_vehicle_1": "Contributing factor",
    },
    height=500,
)
fig.add_vline(x=1, line_dash="dash")
fig.update_yaxes(categoryorder="total ascending")
fig.show()

Factors to the right of the dashed line are more common at night. The top of the list is exactly what you'd guess: **alcohol** and **unsafe speed**. Those are also the kinds of crashes most likely to be deadly, which helps explain why late-night crashes are more often fatal.

In [ ]:
for factor in ["Alcohol Involvement", "Unsafe Speed"]:
    row = factors[factors["contributing_factor_vehicle_1"] == factor]
    if not row.empty:
        row = row.iloc[0]
        print(
            f"{factor}: {row['night_pct']:.1f}% of late-night crashes vs. {row['day_pct']:.1f}% of daytime crashes"
        )

## Conclusion

My hypothesis was that late-night crashes are more dangerous. The answer turned out to be **"yes and no."**

- **Late-night crashes are less likely to injure someone.** That's mostly because there are fewer pedestrians and cyclists out at night, and they make up a big share of the people hurt during the day.
- **But late-night crashes are much more likely to kill someone.** The likely reason is _what kind_ of crashes happen at night: alcohol and speeding show up far more often in late-night crashes than daytime ones.

So in a sense, late-night crashes are fewer and often less risky for bystanders, but when things go wrong, they go very wrong.

## Caveats

- **Crash times are approximate.** Many times are rounded, and I had to drop crashes recorded at exactly midnight because that seems to mean "time unknown."
- **Fatal crashes are rare,** so the hour-by-hour fatality numbers are noisy. Comparing the two periods is more reliable than comparing individual hours.
- **Contributing factors are an officer's judgment** at the scene, and many crashes are marked `Unspecified`.
- This only covers **police-reported** crashes in **2024**. Minor crashes that were never reported aren't included, and other years could look different.
- I'm comparing rates _per crash_, not per mile driven. This tells us how serious a crash is when it happens, not how likely you are to crash in the first place.